# Embedding Evaluation — Notebook

End-to-end walkthrough using . Run all cells top-to-bottom; all figures are interactive (Plotly).

In [9]:
import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path("..").resolve()))

from pai.ag_emb.services.evaluate import run_image2image_eval
from pai.ag_emb.services.reporting import (
    plot_cosine_similarity,
    plot_knn_confusion,
    plot_lle,
    plot_tsne,
    print_result,
)

### Load Data

In [10]:
payload_path = Path("emb_image2image.json")
if not payload_path.exists():
    raise FileNotFoundError("emb_image2image.json not found — start Jupyter from the examples/ directory")

with open(payload_path) as f:
    payload = json.load(f)

embeddings: dict[str, list[float]] = payload["embeddings"]
print(f"Loaded {len(embeddings)} embeddings  (dim={len(next(iter(embeddings.values())))})")

Loaded 23 embeddings  (dim=32)


### Run Evaluation

In [11]:
result = run_image2image_eval(
    image_embeddings=embeddings,
    k_values=[5, 10],
    dataset_root="images",
    sample_pairs=None,
)

print_result(result)

n_items      : 23
embedding_dim: 32
classes      : ['corn', 'soybean']
k_values     : [5, 10]

── global_metrics ──────────────────────────────────────────────────────
  pairwise cosine    : mean=0.5345  std=0.4725  (p05=-0.0237  p50=0.9187  p95=0.9907)
  centroid cosine    : mean=0.7448  std=0.2318  norm=0.7448
  intra/inter gap    : 0.9493  (intra=0.9548  inter=0.0055)
  effective_rank     : 1.22  (ratio=0.0380  dim=32)
  uniformity         : -0.7399

  hubness@5         : mean=5.0000  std=1.9111  p95=7.0000
  hubness@10        : mean=10.0000  std=4.7822  p95=20.8000
  knn_radius@5         : mean=0.9659  std=0.0330  p05=0.9090  p95=0.9900
  knn_radius@10        : mean=0.6538  std=0.4190  p05=0.0020  p95=0.9353
  mean_top_k_sim@5         : mean=0.9789  std=0.0170  p05=0.9452  p95=0.9913
  mean_top_k_sim@10        : mean=0.8518  std=0.1812  p05=0.5724  p95=0.9738
  outlier_score@5         : mean=0.0211  std=0.0170  p95=0.0548
  outlier_score@10        : mean=0.1482  std=0.1812  p95=0.4

### Visualizations

All plots are interactive — hover for details, click legend entries to toggle classes, and drag to rotate 3D views.

#### KNN Confusion Matrix

Rows = true class, columns = neighbor class, values = fraction of k-NN neighbors belonging to each class. The diagonal equals mean KNN purity — higher is better.

In [12]:
plot_knn_confusion(result, output_path=None)

#### Pairwise Cosine Similarity

Full N×N cosine similarity matrix sorted by class. Within-class blocks sit on the diagonal — tighter, brighter blocks indicate a more discriminative embedding space.

In [13]:
plot_cosine_similarity(embeddings, result, output_path=None)

#### t-SNE — 2D

t-SNE preserves local neighborhood structure. Well-separated clusters indicate the model has learned class-discriminative features.

In [14]:
plot_tsne(embeddings, result, output_path=None, dimensions=2)

  t-SNE 2D — fitting 23 samples (perplexity=3, iter=1000)...


#### t-SNE — 3D

3D variant — drag to rotate, scroll to zoom.

In [15]:
plot_tsne(embeddings, result, output_path=None, dimensions=3)

  t-SNE 3D — fitting 23 samples (perplexity=3, iter=2000)...


#### LLE — 3D

Locally Linear Embedding preserves local geometry rather than global distances, complementing the t-SNE view. Drag to rotate.

In [16]:
plot_lle(embeddings, result, output_path=None)

  LLE 3D — fitting 23 samples (n_neighbors=3)...
